# SERP Difficulty via Token Overlap

**Question:** Do SERPs where results look more alike produce different foraging behavior?

**Difficulty metric:** Mean pairwise Jaccard similarity across all result token sets on a SERP. High overlap = results are hard to discriminate = harder SERP.

**Dependent variables:**
1. Trial duration (first event to click)
2. Scroll regression count
3. Click position (do users satisfice earlier on hard SERPs?)
4. Fixation count (total investment)
5. Page coverage (how far down do they scan?)
6. Per-result dwell curve shape (flat vs declining)

**Key difference from serp_priming.ipynb:** That notebook asked whether overlap predicts *speed* at each position (priming hypothesis — null result). This notebook asks whether SERP-level difficulty changes the *strategy*.

**Dataset:** AdSERP — 2,776 trials, 47 participants, product purchase queries. Each participant gets a unique SERP snapshot from Google, so difficulty varies even within the same query.

In [ ]:
# Shared data loading — see data_loader.py for all utilities
from data_loader import *
setup_plotting()
import os, re, csv, json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


## Step 1: Extract SERP text and compute difficulty

Reuses the tokenization from serp_priming.ipynb. Difficulty = mean pairwise Jaccard across all result pairs on a SERP.

### Key Measures

| Measure | Definition | Units | Interpretation |
|---------|-----------|-------|----------------|
| SERP difficulty | Mean pairwise Jaccard similarity across all result token sets on a SERP | ratio (0-1) | High overlap = results look alike = harder to discriminate |
| Dependent variables | Trial duration, regression count, click position, fixation count, page coverage, dwell curve shape | various | Tested against difficulty; null or weak effects after controlling for individual differences |


In [ ]:
STOPWORDS = set('the a an and or but in on at to for of is it this that was were be been '
                'being have has had do does did will would shall should may might can could '
                'with from by as are not no its my your his her their our its '
                'between both during each few how more most other some such through until '
                'where which while who whom why into over under'.split())

def tokenize(text):
    """Simple whitespace + punctuation tokenizer, lowercased, stopwords removed."""
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def extract_serp_results(html_path):
    """Extract ordered list of search results from a Google SERP HTML file."""
    with open(html_path, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    results = []
    rso = soup.find(id='rso')
    if not rso:
        rso = soup
    
    for i, h3 in enumerate(rso.find_all('h3')):
        title_text = h3.get_text(strip=True)
        container = h3.parent
        for _ in range(5):
            if container and container.parent:
                container = container.parent
                if container.name == 'div':
                    if container.get('class') and any('g' in c for c in container.get('class', [])):
                        break
        
        snippet_parts = []
        if container:
            for text_elem in container.find_all(['span', 'div'], recursive=True):
                text = text_elem.get_text(strip=True)
                if text and text != title_text and len(text) > 20:
                    snippet_parts.append(text)
        
        link = h3.find_parent('a')
        url = link.get('href', '') if link else ''
        
        all_text = title_text + ' ' + ' '.join(snippet_parts[:3])
        tokens = tokenize(all_text)
        
        results.append({
            'position': i,
            'title': title_text,
            'tokens': tokens,
            'token_set': set(tokens),
        })
    
    return results

def compute_serp_difficulty(results):
    """Mean pairwise Jaccard similarity across all result pairs.
    
    Higher = more overlap = harder to discriminate.
    """
    pairwise = []
    for i in range(len(results)):
        for j in range(i + 1, len(results)):
            ti = results[i]['token_set']
            tj = results[j]['token_set']
            union = ti | tj
            if union:
                pairwise.append(len(ti & tj) / len(union))
    return np.mean(pairwise) if pairwise else 0.0

# Process all SERPs
print('Extracting SERP text and computing difficulty...')
serp_difficulty = {}  # trial_id -> difficulty score
serp_n_results = {}   # trial_id -> number of results
errors = 0

for html_path in serp_files:
    tid = html_path.stem
    try:
        results = extract_serp_results(html_path)
        if len(results) >= 3:
            serp_difficulty[tid] = compute_serp_difficulty(results)
            serp_n_results[tid] = len(results)
    except:
        errors += 1

diffs = np.array(list(serp_difficulty.values()))
print(f'Processed: {len(serp_difficulty)} SERPs ({errors} errors)')
print(f'Difficulty: mean={diffs.mean():.3f}, median={np.median(diffs):.3f}, std={diffs.std():.3f}')
print(f'Range: [{diffs.min():.3f}, {diffs.max():.3f}]')

## Step 2: Link difficulty to behavioral measures

For each trial with both SERP difficulty and behavioral data, collect DVs from the catalog and raw data files.

In [ ]:
def get_click_position(trial_id):
    """Get Y-position of first click from mouse data via data_loader."""
    _, _, clicks = load_mouse_events(trial_id)
    return clicks[0][2] if clicks else None

def count_regressions(trial_id):
    """Count scroll regressions (upward scrolls after a downward scroll)."""
    path = os.path.join(MOUSE_DIR, f'{trial_id}.csv')
    if not os.path.exists(path):
        return None
    scrolls = []
    with open(path) as f:
        for row in csv.DictReader(f):
            if row['event'] == 'scroll':
                scrolls.append(float(row['ypos']))
    if len(scrolls) < 2:
        return 0
    regressions = 0
    prev_direction = None
    for i in range(1, len(scrolls)):
        dy = scrolls[i] - scrolls[i-1]
        if dy < -5:  # upward scroll (regression)
            direction = 'up'
        elif dy > 5:
            direction = 'down'
        else:
            continue
        if direction == 'up' and prev_direction == 'down':
            regressions += 1
        prev_direction = direction
    return regressions

# Build linked dataset
trials = []
for tid, diff in serp_difficulty.items():
    meta = trial_meta.get(tid)
    if not meta:
        continue
    
    click_y = get_click_position(tid)
    n_regressions = count_regressions(tid)
    
    if meta['duration_s'] < 1 or meta['duration_s'] > 120:
        continue
    
    trials.append({
        'trial_id': tid,
        'participant': meta['participant'],
        'difficulty': diff,
        'n_results': serp_n_results[tid],
        'duration_s': meta['duration_s'],
        'fixation_count': meta['fixation_count'],
        'page_coverage_pct': meta['page_coverage_pct'],
        'n_regressions': n_regressions if n_regressions is not None else 0,
        'has_regression': meta['has_scroll_regression'],
        'click_y': click_y,
        'scroll_count': meta['scroll_event_count'],
    })

print(f'Linked trials: {len(trials)}')

# Convert to arrays for easy slicing
diff_arr = np.array([t['difficulty'] for t in trials])
dur_arr = np.array([t['duration_s'] for t in trials])
fix_arr = np.array([t['fixation_count'] for t in trials])
cov_arr = np.array([t['page_coverage_pct'] for t in trials])
reg_arr = np.array([t['n_regressions'] for t in trials])
click_arr = np.array([t['click_y'] if t['click_y'] else np.nan for t in trials])

## Step 3: Difficulty distribution and correlations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1: Difficulty distribution
ax = axes[0, 0]
ax.hist(diff_arr, bins=50, color='#4ecdc4', alpha=0.8, edgecolor='white')
ax.axvline(np.median(diff_arr), color='#dc2626', linestyle='--', label=f'median={np.median(diff_arr):.3f}')
ax.set_xlabel('SERP Difficulty (mean pairwise Jaccard)')
ax.set_ylabel('Count')
ax.set_title('Difficulty Distribution')
ax.legend()

# Correlations for remaining 5 panels
dvs = [
    ('Duration (s)', dur_arr, '#e07c24'),
    ('Fixation Count', fix_arr, '#7c3aed'),
    ('Page Coverage (%)', cov_arr, '#16a34a'),
    ('Regression Count', reg_arr, '#dc2626'),
    ('Click Y Position', click_arr, '#2563eb'),
]

for idx, (label, arr, color) in enumerate(dvs):
    ax = axes[(idx + 1) // 3, (idx + 1) % 3]
    mask = np.isfinite(arr)
    x, y = diff_arr[mask], arr[mask]
    
    ax.scatter(x, y, alpha=0.08, s=8, color=color, rasterized=True)
    
    # Regression line
    r, p = stats.spearmanr(x, y)
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color='black', linewidth=1.5)
    
    ax.set_xlabel('SERP Difficulty')
    ax.set_ylabel(label)
    ax.set_title(f'{label}\n\u03c1 = {r:.3f}, p = {p:.2e}')

plt.suptitle('SERP Difficulty (Token Overlap) vs Foraging Behavior', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plot_difficulty_overview.png', dpi=200, bbox_inches='tight')
plt.show()

## Step 4: Tercile comparison

Split SERPs into easy / medium / hard by difficulty terciles. Compare behavioral profiles.

In [ ]:
t1, t2 = np.percentile(diff_arr, [33, 66])
groups = {
    'Easy (low overlap)': [t for t in trials if t['difficulty'] <= t1],
    'Medium': [t for t in trials if t1 < t['difficulty'] <= t2],
    'Hard (high overlap)': [t for t in trials if t['difficulty'] > t2],
}

print(f'Tercile boundaries: {t1:.3f}, {t2:.3f}')
print(f'Group sizes: {" / ".join(f"{k}: {len(v)}" for k, v in groups.items())}\n')

metrics = ['duration_s', 'fixation_count', 'page_coverage_pct', 'n_regressions']
metric_labels = ['Duration (s)', 'Fixation Count', 'Page Coverage (%)', 'Regressions']

print(f'{"":>22s}  {"Easy":>10s}  {"Medium":>10s}  {"Hard":>10s}   KW p-value')
print('-' * 72)

for metric, label in zip(metrics, metric_labels):
    vals = {name: [t[metric] for t in group] for name, group in groups.items()}
    means = {name: np.mean(v) for name, v in vals.items()}
    
    # Kruskal-Wallis test (non-parametric)
    h_stat, p_val = stats.kruskal(*vals.values())
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    
    m = list(means.values())
    print(f'{label:>22s}  {m[0]:10.2f}  {m[1]:10.2f}  {m[2]:10.2f}   {p_val:.4f} {sig}')

# Click Y position (has NaNs)
vals_click = {name: [t['click_y'] for t in group if t['click_y'] is not None] 
              for name, group in groups.items()}
means_click = {name: np.mean(v) for name, v in vals_click.items()}
h, p = stats.kruskal(*vals_click.values())
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
m = list(means_click.values())
print(f'{"Click Y Position":>22s}  {m[0]:10.1f}  {m[1]:10.1f}  {m[2]:10.1f}   {p:.4f} {sig}')

# Regression rate (proportion with at least one regression)
for name, group in groups.items():
    rate = np.mean([t['has_regression'] for t in group])
    print(f'  {name} regression rate: {rate:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = ['#4ecdc4', '#e07c24', '#dc2626']
labels = list(groups.keys())

for idx, (metric, mlabel) in enumerate(zip(metrics, metric_labels)):
    ax = axes[idx]
    data = [[t[metric] for t in groups[name]] for name in labels]
    
    parts = ax.violinplot(data, positions=[1, 2, 3], showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.6)
    
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Easy', 'Medium', 'Hard'], fontsize=10)
    ax.set_ylabel(mlabel)
    ax.set_title(mlabel)

plt.suptitle('Behavioral Profiles by SERP Difficulty Tercile', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plot_difficulty_terciles.png', dpi=200, bbox_inches='tight')
plt.show()

## Step 5: Confound check — difficulty vs number of results

SERPs with more results mechanically have higher pairwise overlap (more token sharing opportunities). Check if difficulty is just a proxy for result count.

In [ ]:
nresults_arr = np.array([t['n_results'] for t in trials])

r_dn, p_dn = stats.spearmanr(diff_arr, nresults_arr)
print(f'Difficulty x n_results: rho = {r_dn:.3f}, p = {p_dn:.2e}')

# Partial correlation: difficulty -> DV, controlling for n_results
# Using residualization approach
def partial_spearman(x, y, z):
    """Spearman correlation between x and y, controlling for z."""
    # Rank-transform then residualize
    rx = stats.rankdata(x)
    ry = stats.rankdata(y)
    rz = stats.rankdata(z)
    
    # Residualize x and y against z
    slope_xz = np.polyfit(rz, rx, 1)[0]
    slope_yz = np.polyfit(rz, ry, 1)[0]
    rx_resid = rx - slope_xz * rz
    ry_resid = ry - slope_yz * rz
    
    return stats.pearsonr(rx_resid, ry_resid)

print('\nPartial correlations (difficulty -> DV, controlling for n_results):')
for label, arr in [('Duration', dur_arr), ('Fixations', fix_arr), 
                    ('Coverage', cov_arr), ('Regressions', reg_arr)]:
    r, p = partial_spearman(diff_arr, arr, nresults_arr)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  {label:>12s}: r = {r:.4f}, p = {p:.4f} {sig}')

# Click Y with NaN handling
mask = np.isfinite(click_arr)
r, p = partial_spearman(diff_arr[mask], click_arr[mask], nresults_arr[mask])
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f'  {"Click Y":>12s}: r = {r:.4f}, p = {p:.4f} {sig}')

## Step 6: Within-participant effects

Each participant sees 10 SERPs of varying difficulty. Does difficulty predict behavior *within* each participant, not just across participants?

In [ ]:
# Group trials by participant
by_participant = defaultdict(list)
for t in trials:
    by_participant[t['participant']].append(t)

# Per-participant correlations between difficulty and DVs
within_corrs = defaultdict(list)  # metric -> [per-participant rho]

for pid, ptrials in by_participant.items():
    if len(ptrials) < 5:  # need enough trials for meaningful correlation
        continue
    
    pdiff = [t['difficulty'] for t in ptrials]
    
    for metric in ['duration_s', 'fixation_count', 'n_regressions', 'page_coverage_pct']:
        vals = [t[metric] for t in ptrials]
        if np.std(pdiff) > 0 and np.std(vals) > 0:
            r, _ = stats.spearmanr(pdiff, vals)
            if np.isfinite(r):
                within_corrs[metric].append(r)

print('Within-participant correlations (difficulty -> DV):')
print(f'{"Metric":>20s}  {"Mean rho":>9s}  {"t":>7s}  {"p":>9s}  N')
print('-' * 60)

for metric, label in zip(['duration_s', 'fixation_count', 'n_regressions', 'page_coverage_pct'],
                          ['Duration', 'Fixations', 'Regressions', 'Coverage']):
    corrs = within_corrs[metric]
    mean_r = np.mean(corrs)
    # One-sample t-test: is mean rho different from 0?
    t_stat, p_val = stats.ttest_1samp(corrs, 0)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f'{label:>20s}  {mean_r:9.3f}  {t_stat:7.2f}  {p_val:9.4f}  {len(corrs)} {sig}')

## Summary

Interpret results above. Key questions:
1. Does SERP difficulty (token overlap) predict *any* behavioral measure at the between-trial level?
2. Do the effects survive controlling for number of results?
3. Are effects visible within-participant (ruling out individual-difference confounds)?

If effects are null at bag-of-words Jaccard, next step is semantic similarity (embeddings) per the TODO.